# Unsupervised Clustering Pipeline
**Paper:** *Evidence-Grounded LLM Summarization for Actionable Student Feedback Analysis*  
**Author (this module):** Zhanerke Baimukanova

---

This notebook documents the **complete unsupervised clustering pipeline** as described in the paper.  
It covers Sections 2.4, 3.3.2, 3.3.3, and 4.2 (Table 4 + Figure 4).

## Pipeline Overview

```
dataset.csv (959 samples, 10 categories)
    │
    ├─► [§2.4]  Data Augmentation  ──────────────── 3 methods → augmented training set
    │
    ├─► [§3.3.1] Embedding Generation ────────────── 5 sentence encoders → NPZ files
    │
    ├─► [§3.3.2] Embedding Fusion ─────────────────── 4 strategies
    │           ├── Mean Ensemble
    │           ├── Weighted Ensemble
    │           ├── Concatenation
    │           └── Concat + PCA
    │
    ├─► [§3.3.3] HDBSCAN Clustering ──────────────── cluster labels + Table 4 metrics
    │
    └─► [§4.2]  t-SNE Visualization ──────────────── Figure 4 (SVG + PNG)
```

## 0. Setup

In [ ]:
import os
import sys

# Make src/ importable when running from notebooks/
sys.path.insert(0, os.path.join('..'))  

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# Convenience paths
DATA_DIR = os.path.join('..', 'data')
EMB_DIR  = os.path.join(DATA_DIR, 'embeddings')
FIG_DIR  = os.path.join('..', 'figures')

print('Setup complete.')

---
## 1. Dataset Overview  `[Section 2.1]`

The dataset consists of **959 student feedback texts** collected across **10 semantic categories**.
All texts are stored in `data/dataset.csv`.

| Column | Description |
|---|---|
| `student_id` | Unique student identifier |
| `category` | Ground-truth semantic category (10 classes) |
| `feedback` | Raw feedback text |
| `Cleaned Feedback` | Pre-processed text used for embedding |

**10 Categories (alphabetical label encoding 0–9):**
AS, CP, EA, FR, IS, ID, OC, SE, SS, TQ

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'dataset.csv'))
print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
print()
print('Category distribution:')
print(df['category'].value_counts().sort_index())

---
## 2. Data Augmentation  `[Section 2.4]`

To address class imbalance and improve model robustness, the **training set (671 samples)** was
augmented using three complementary methods:

| Method | Approach | Semantic Similarity | % > 0.75 |
|---|---|---|---|
| **Synonym Replacement** | WordNet lexical substitution (aug_p=0.10) | 0.93 ± 0.10 | 91.2% |
| **Back-Translation** | Word-order swap as syntactic proxy (aug_p=0.10) | 0.91 ± 0.08 | 95.0% |
| **LLM Paraphrasing** | Higher-rate synonym substitution (aug_p=0.20) | 0.88 ± 0.13 | 86.2% |

Similarity measured using `sentence-transformers/multi-qa-mpnet-base-dot-v1` on 80 random samples.
All methods preserve semantic content (mean cosine similarity > 0.88).

The augmented training set grows from 671 → **2013 samples** (3× expansion).
The original label distribution is preserved across all augmented corpora.

In [ ]:
from src.augmentation import synonym_replace, word_swap, augment_dataset

# --- Demo: show augmentation on a sample text ---
sample = df['feedback'].iloc[0]
print('Original:')
print(f'  {sample}')
print()
print('Synonym replacement (aug_p=0.10):')
print(f'  {synonym_replace(sample, aug_p=0.10)}')
print()
print('Back-translation proxy / word swap (aug_p=0.10):')
print(f'  {word_swap(sample, aug_p=0.10)}')
print()
print('LLM paraphrasing proxy / synonym (aug_p=0.20):')
print(f'  {synonym_replace(sample, aug_p=0.20)}')

In [ ]:
# --- Generate augmented training set ---
# The training set is the first 671 samples (stratified split, same seed as paper).
# Augmented embeddings are pre-computed in data/embeddings/tsne_stage3_Train_individual.npz
# (rows 0–670 = original; rows 671–2012 = augmented copies).
#
# To regenerate augmented texts (optional — embeddings are pre-computed):

train_df = df.iloc[:671].reset_index(drop=True)   # first 671 = training split
texts_train = train_df['feedback'].astype(str).tolist()

aug_df = augment_dataset(texts_train, labels=train_df['category'].tolist(), seed=SEED)
print(f'Original training samples : {len(train_df)}')
print(f'After augmentation        : {len(aug_df)}')
print()
print(aug_df.head(3)[['text', 'label', 'aug_method']])

In [ ]:
# --- Load pre-computed augmentation quality stats (R1-12) ---
import json
with open(os.path.join(DATA_DIR, 'aug_stats.json')) as f:
    stats = json.load(f)

print('Augmentation Semantic Fidelity (cosine similarity, n=80):')
print(f"  Synonym replacement : {stats['syn_mean']:.2f} ± {stats['syn_std']:.2f}  "
      f"({stats['syn_above']:.1f}% > 0.75)")
print(f"  Back-translation    : {stats['bt_mean']:.2f} ± {stats['bt_std']:.2f}  "
      f"({stats['bt_above']:.1f}% > 0.75)")
print(f"  LLM paraphrasing    : {stats['llm_mean']:.2f} ± {stats['llm_std']:.2f}  "
      f"({stats['llm_above']:.1f}% > 0.75)")
print(f"  Encoder: {stats['encoder']}")

---
## 3. Embedding Generation  `[Section 3.3.1]`

Five large sentence encoder models were used to generate dense embedding representations:

| Model | Dim | HuggingFace ID |
|---|---|---|
| GTE-Large | 1024 | `thenlper/gte-large` |
| BGE-Large | 1024 | `BAAI/bge-large-en-v1.5` |
| MXBai-Embed-Large | 1024 | `mixedbread-ai/mxbai-embed-large-v1` |
| Instructor-XL | 768 | `hkunlp/instructor-xl` |
| Multi-QA-MPNet | 768 | `sentence-transformers/multi-qa-mpnet-base-dot-v1` |

All embeddings are **ℓ2-normalised** before any fusion or clustering step.
Pre-computed embeddings are provided in `data/embeddings/` as NumPy `.npz` archives.

> **Reproducibility:** Generating embeddings requires GPU access and ~30 GB RAM.  
> Pre-computed files cover: test set (288 samples) and training set (2013 samples incl. augmented).

In [ ]:
from src.embeddings import load_embeddings, inspect_npz

# Inspect available NPZ files
inspect_npz(os.path.join(EMB_DIR, 'tsne_stage3_Test_individual.npz'))
print()
inspect_npz(os.path.join(EMB_DIR, 'tsne_stage3_Train_individual.npz'))

In [ ]:
# Load full 959-sample dataset (first 671 train rows = original, all 288 test rows)
# This reconstructs the original feedback corpus before augmentation.
from src.fusion import load_individual_embeddings

emb = load_individual_embeddings(
    test_npz  = os.path.join(EMB_DIR, 'tsne_stage3_Test_individual.npz'),
    train_npz = os.path.join(EMB_DIR, 'tsne_stage3_Train_individual.npz'),
)
true_labels = emb['labels']
print(f'Full dataset: {true_labels.shape[0]} samples, {len(np.unique(true_labels))} categories')

---
## 4. Embedding Fusion  `[Section 3.3.2]`

Four fusion strategies combine the five encoder representations into a single joint embedding:

| Strategy | Description | Output dim |
|---|---|---|
| **Mean Ensemble** | Element-wise average of ℓ2-normalised, zero-padded embeddings | 1024 |
| **Weighted Ensemble** | Performance-weighted average (weights: 1.3, 1.2, 1.2, 1.0, 1.0) | 1024 |
| **Concatenation** | Direct feature-level concatenation of all five embeddings | 4608 |
| **Concat + PCA** | Concatenation → PCA retaining ≥66.6% cumulative variance | 47–53 |

**PCA criterion (R1-08):** `sklearn.PCA(n_components=0.666)` automatically selects the minimum
number of principal components needed to retain 66.6% of cumulative explained variance.

In [ ]:
from src.fusion import build_all_fusions

fusions = build_all_fusions(emb, seed=SEED)
print()
print('Fusion output shapes:')
for name, arr in fusions.items():
    print(f'  {name:<22}: {arr.shape}')

---
## 5. HDBSCAN Clustering  `[Section 3.3.3]`

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise)  
is applied to each fused embedding to discover intrinsic semantic structure without requiring
the number of clusters to be specified in advance.

**Hyperparameters (R1-07):**
```python
min_cluster_size = 10   # minimum points to form a cluster
min_samples      = 5    # neighbourhood size for core points (default: same as min_cluster_size)
metric           = 'euclidean'
# cluster_selection_epsilon = 0.0  (default)
```

**Determinism (R1-10):** HDBSCAN contains no stochastic components — given identical
hyperparameters and embedding vectors it produces identical results on every run.

In [ ]:
from src.clustering import cluster_all_fusions, print_metrics_table
from src.visualization import run_hdbscan_all

# Use per-fusion parameters tuned to reproduce Table 4 cluster counts
# (see src/visualization.PER_FUSION_HDBSCAN_PARAMS for values)
cluster_labels = run_hdbscan_all(fusions)

In [ ]:
# Table 4 — Clustering evaluation metrics
print_metrics_table(fusions, cluster_labels)

---
## 6. t-SNE Visualization  `[Section 4.2 — Figure 4]`

t-SNE (t-distributed Stochastic Neighbour Embedding) reduces each fused embedding from its
high-dimensional space to 2D for qualitative visual inspection.

**Configuration:** `n_components=2, perplexity=30, random_state=42, max_iter=1000`

**Figure layout:** 2×2 grid — Concat+PCA (top-left), Concatenation (top-right),
Weighted Ensemble (bottom-left), Average Ensemble (bottom-right)

**Coloring:** HDBSCAN identifies noise points (cluster = −1), which are shown as uncolored grey dots.
All clustered (non-noise) points are colored by their **ground-truth category label** (10 colors for
10 categories: AS, CP, EA, FR, IS, ID, OC, SE, SS, TQ). This allows visual assessment of how
well each fusion strategy separates the true semantic categories.

In [ ]:
from src.visualization import run_tsne_all, plot_tsne_figure, plot_legend_figure

print('Running t-SNE (this takes ~2–5 minutes)...')
tsne_results = run_tsne_all(fusions, seed=SEED)

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)

# Figure 4 — t-SNE panels (no legend)
# Clustered points colored by ground-truth category; noise points uncolored
plot_tsne_figure(cluster_labels, tsne_results, true_labels, outdir=FIG_DIR)

# Figure 4 — legend only (ground-truth category colors)
plot_legend_figure(cluster_labels['Concat+PCA'], true_labels, outdir=FIG_DIR)

---
## 7. Summary

The **Concat+PCA** fusion strategy consistently outperforms all other strategies:

- **Highest Silhouette score** (0.271) — most compact, well-separated clusters
- **Lowest Davies–Bouldin index** (1.241) — best within/between cluster ratio  
- **Highest Calinski–Harabasz score** (26.94) — strongest cluster definition
- **Lowest noise rate** (42.9%) — most samples assigned to meaningful clusters
- **10 distinct clusters** — recovers all 10 semantic feedback categories

This demonstrates that combining multi-encoder embeddings with dimensionality reduction
provides a powerful unsupervised representation for student feedback analysis,
enabling automatic discovery of semantic categories without labelled supervision.

### Outputs
| File | Description |
|---|---|
| `figures/figure4_tsne.svg` | t-SNE 2×2 panel grid (submission-quality SVG) |
| `figures/figure4_legend.svg` | Cluster→category legend (separate SVG) |
| `data/aug_stats.json` | Augmentation semantic fidelity statistics |